In [23]:
#Step 2: Verify Repositories with REST API
# This script checks repositories from Step 1 against the GitHub REST API.
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    print(f"🔁 Using token GITHUB_TOKEN_{token_index + 1} of {len(tokens)}")
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step1_search_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
failed_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_failed_requests.csv"

# === Load latest data ===
if os.path.exists(output_path):
    print("♻️ Resuming from previously saved output...")
    df = pd.read_csv(output_path)
else:
    print("📥 Starting fresh from Step 1 output...")
    df = pd.read_csv(input_path)
    df["Valid_Repo_Step2"] = "none"
    df["rest_check_status"] = ""
    df["rest_reason"] = ""

# === Filter: outstanding repos only ===
outstanding_df = df[(df["Valid_Repo_Step1"] == "yes") & (df["Valid_Repo_Step2"] == "none")].copy()
print(f"🔍 Reviewing {len(outstanding_df)} outstanding repos...\n")

failed_repos = []
valid_langs = ["Java", "Kotlin", "Dart"]
MAX_RETRIES = 3

# === Process each outstanding repo ===
for idx, row in enumerate(outstanding_df.itertuples(), start=1):
    repo = row.full_name
    url = f"https://api.github.com/repos/{repo}"

    print(f"🔎 [{idx}/{len(outstanding_df)}] Checking: {repo}")
    retries = 0
    while retries < MAX_RETRIES:
        response = requests.get(url, headers=get_headers())
        if response.status_code == 403:
            print(f"⏳ Rate limit hit. Sleeping 10s...")
            sleep(10)
            retries += 1
            continue
        break

    if response.status_code != 200:
        print(f"❌ Failed: {repo} - HTTP {response.status_code}")
        df.loc[df["full_name"] == repo, "Valid_Repo_Step2"] = "no"
        df.loc[df["full_name"] == repo, "rest_check_status"] = "reject"
        df.loc[df["full_name"] == repo, "rest_reason"] = f"HTTP {response.status_code}"
        failed_repos.append({"full_name": repo, "error": response.status_code})
        df.to_csv(output_path, index=False)
        sleep(1)
        continue

    # === Apply filtering logic ===
    data = response.json()
    reasons = []
    if data.get("fork", True): reasons.append("fork")
    if data.get("archived", True): reasons.append("archived")
    if data.get("stargazers_count", 0) <= 50: reasons.append("low stars")
    if data.get("language") not in valid_langs: reasons.append("language mismatch")

    if reasons:
        df.loc[df["full_name"] == repo, "Valid_Repo_Step2"] = "no"
        df.loc[df["full_name"] == repo, "rest_check_status"] = "reject"
        df.loc[df["full_name"] == repo, "rest_reason"] = ", ".join(reasons)
    else:
        df.loc[df["full_name"] == repo, "Valid_Repo_Step2"] = "yes"
        df.loc[df["full_name"] == repo, "rest_check_status"] = "pass"
        df.loc[df["full_name"] == repo, "rest_reason"] = ""

    # === Save interim result ===
    df.to_csv(output_path, index=False)

# === Save failures
if failed_repos:
    pd.DataFrame(failed_repos).to_csv(failed_path, index=False)
    print(f"\n⚠️ Failed requests saved to: {failed_path}")

print(f"\n✅ Step 2 complete. Verified output saved to: {output_path}")


📥 Starting fresh from Step 1 output...
🔍 Reviewing 51699 outstanding repos...

🔎 [1/51699] Checking: ligi/gobandroid
🔁 Using token GITHUB_TOKEN_1 of 6
🔎 [2/51699] Checking: quran/quran_android
🔁 Using token GITHUB_TOKEN_2 of 6
🔎 [3/51699] Checking: evanchooly/javabot
🔁 Using token GITHUB_TOKEN_3 of 6
🔎 [4/51699] Checking: facebook/facebook-android-sdk
🔁 Using token GITHUB_TOKEN_4 of 6
🔎 [5/51699] Checking: thewca/tnoodle
🔁 Using token GITHUB_TOKEN_5 of 6
🔎 [6/51699] Checking: dkandalov/pomodoro-tm
🔁 Using token GITHUB_TOKEN_6 of 6
🔎 [7/51699] Checking: dkandalov/scratch
🔁 Using token GITHUB_TOKEN_1 of 6
🔎 [8/51699] Checking: Bombe/Sone
🔁 Using token GITHUB_TOKEN_2 of 6
🔎 [9/51699] Checking: thunderbird/thunderbird-android
🔁 Using token GITHUB_TOKEN_3 of 6
🔎 [10/51699] Checking: JetBrains/ideavim
🔁 Using token GITHUB_TOKEN_4 of 6
🔎 [11/51699] Checking: wuan/bo-android
🔁 Using token GITHUB_TOKEN_5 of 6
🔎 [12/51699] Checking: UweTrottmann/SeriesGuide
🔁 Using token GITHUB_TOKEN_6 of 6
🔎 [1